# Creating the Prediction pipeline

## Module Loading

In [1]:
from google.colab import drive
from shutil import copy2
from duckdb import connect as dcon
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
import plotly.express as px
from pylab import rcParams
import pandas as pd
import numpy as np
from warnings import filterwarnings

%matplotlib inline
darkmodel = True
rcParams['figure.figsize'] = (12,6)
pd.options.display.float_format = '{:,.2f}'.format
filterwarnings('ignore', category=FutureWarning)

In [2]:
if darkmodel:
    # 1. Define a sophisticated E-commerce color palette
    # These colors are chosen for high contrast against the #212946 background
    colors = [
        "#08F7FE",  # Cyan Glow
        "#FE53BB",  # Neon Pink
        "#F5D300",  # Cyber Yellow
        "#00ff41",  # Matrix Green
        "#9467bd",  # Royal Purple
    ]

    # 2. Enhanced Dictionary with Complex Styling
    refined_dark_style = {
        # Background and Canvas
        "figure.facecolor": "#212946",
        "axes.facecolor": "#212946",
        "savefig.facecolor": "#212946",

        # Grid Sophistication
        "axes.grid": True,
        "axes.grid.which": "both",
        "grid.color": "#2A3459",
        "grid.linewidth": "1",
        "grid.alpha": 0.5,

        # Typography & Labels (Optimized for readability)
        "text.color": "#E2E2E2",
        "axes.labelcolor": "#E2E2E2",
        "axes.labelsize": 14,
        "axes.titlesize": 18,
        "axes.titleweight": "bold",
        "axes.titlepad": 20,
        "xtick.color": "#8E9CC3",
        "ytick.color": "#8E9CC3",
        "font.size": 12,

        # Spines (Clean aesthetic)
        "axes.spines.left": False,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.spines.bottom": True,
        "axes.edgecolor": "#2A3459",

        # Line & Marker Settings
        "lines.linewidth": 2.5,
        "lines.markersize": 8,
        "axes.prop_cycle": plt.cycler(color=colors),
    }

    plt.rcParams.update(refined_dark_style)

else:
    # 1. Defining the "Paper & Ink" Palette
    # Deep Blue #003366 | Oxide Red #A52A2A
    ecom_vintage_colors = [
        "#003366",  # Oxford Blue (Primary)
        "#A52A2A",  # Oxide Red (Comparison)
        "#006400",  # Dark Green (Success Metrics)
        "#704214",  # Sepia (Neutral)
    ]

    vintage_style = {
        # Background - The specific parchment hex you requested
        "figure.facecolor": "#f7e4b7",
        "axes.facecolor": "#f7e4b7",
        "savefig.facecolor": "#f7e4b7",

        # Grid - Subtle contrast using a darker version of the background
        "axes.grid": True,
        "grid.color": "#e2d1a8",
        "grid.linestyle": "-",
        "grid.linewidth": 1.0,

        # Typography - Deep Charcoal/Blue instead of pure black for a softer feel
        "text.color": "#2C2C2C",
        "axes.labelcolor": "#2C2C2C",
        "xtick.color": "#5D5D5D",
        "ytick.color": "#5D5D5D",
        "axes.titlesize": 16,
        "axes.titleweight": "bold",
        "axes.titlepad": 15,
        "font.size": 11,

        # Spines - Classic 'L-frame' for publication
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.spines.left": True,
        "axes.spines.bottom": True,
        "axes.edgecolor": "#2C2C2C",
        "axes.linewidth": 1.5,

        # Data Point Styling
        "axes.prop_cycle": plt.cycler(color=ecom_vintage_colors),
        "lines.linewidth": 2.2,
        "lines.markersize": 8,
        "patch.edgecolor": "#f7e4b7", # Borders on bars/pie slices
    }

    plt.rcParams.update(vintage_style)

## Data Test Loading

In [3]:
drive.mount('/content/drive')
pd.set_option('display.max_rows', 50)

def ListFiles(Dirs):
    errormsg = f"Error: Directory '{Dirs}' does not exist or is not a directory."
    assert os.path.isdir(Dirs), errormsg
    file_data = list()
    for item in os.listdir(Dirs):
        item_path = os.path.join(Dirs, item)
        if os.path.isfile(item_path):
            try:
                size_bytes = os.path.getsize(item_path)
                size_mb = size_bytes / (1024 * 1024)  # Convert bytes to MB
                file_data.append({'File Name': item, 'Size (MB)': size_mb})
            except Exception as err:
                print(f"Could not get size for {item_path}: {err}")

    Files = pd.DataFrame(file_data)
    return Files

Mounted at /content/drive


In [4]:
MyFiles = ListFiles('/content/drive/MyDrive/Colab Notebooks')
DatFilename = MyFiles[~MyFiles['File Name'].str.contains('.ipynb', na = False)]
display(DatFilename)

,File Name,Size (MB)
76,MasterData.parquet,145.58
79,InstaCart.db,263.76
82,xgboost_ltr_model.json,2.43
83,catboost_ltr_model.cbm,0.32
84,lightgbm_ltr_model.txt,6.21
85,lgbm_optuna_ranker_model.txt,0.58
86,test_df.parquet,3.84


In [5]:
test_df = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/test_df.parquet')
display(test_df.head())

,order_id,user_id,product_id,aisle_id,department_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,user_total_orders,user_avg_days_between,user_avg_cart_pos,user_total_reorders,index,prod_total_reorders,prod_reorder_rate,prod_order_count,prod_avg_cart_pos
0,252513,110001,41588,14,20,38,4,19,3.00,8,1,38,3.00,6.50,9,32736,256,0.65,391,8.82
1,252513,110001,49628,120,16,38,4,19,3.00,12,1,38,3.00,6.50,9,39078,123,0.66,186,8.49
2,252654,42214,23178,98,7,10,5,16,12.00,3,0,10,12.00,2.00,1,18299,108,0.61,176,7.26
3,252654,42214,24838,91,16,10,5,16,12.00,1,0,10,12.00,2.00,1,19563,1615,0.74,2169,6.28
4,252654,42214,46667,83,4,10,5,16,12.00,2,1,10,12.00,2.00,1,36739,1241,0.62,2007,9.49


In [6]:
db_path = '/content/drive/MyDrive/Colab Notebooks/InstaCart.db'
con = dcon(database=db_path, read_only=True)
tables = con.execute("PRAGMA show_tables;").fetchdf()
print("Tables in InstaCart.db:")
display(tables)

Tables in InstaCart.db:


,name
0,AISLE
1,DepartmentData
2,FullTrainData
3,OrderTest
4,OrderTrain
5,OrdersDetails
6,ProductsData


In [7]:
for table_name in tables['name']:
    print(f"\n--- Sample from Table: {table_name} ---")
    query = f"SELECT * FROM {table_name} LIMIT 3;"
    df_sample = con.execute(query).fetchdf()
    display(df_sample)


--- Sample from Table: AISLE ---


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars



--- Sample from Table: DepartmentData ---


,department_id,department
0,1,frozen
1,2,other
2,3,bakery



--- Sample from Table: FullTrainData ---


...

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0



--- Sample from Table: OrdersDetails ---


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.00
2,473747,1,prior,3,3,12,21.00



--- Sample from Table: ProductsData ---


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7


In [8]:
Q3 = f"SELECT * FROM ProductsData;"
DataProduct = con.execute(Q3).fetchdf()

## Forming Predict Function

### Predicted only which item been ever bought

In [19]:
import xgboost as xgb
import numpy as np

def load_xgb(model_path: str):
    assert os.path.exists(model_path), f"Model file '{model_path}' does not exist."
    model = xgb.Booster()
    model.load_model(model_path)
    return model


In [20]:
import xgboost as xgb
import pandas as pd
from typing import List, Dict

def process_single_user_ltr(user_id: int,
                        full_dataset: pd.DataFrame,
                        bst: xgb.Booster,
                        features: List[str],
                       ):
    """Helper function to process one user at a time for parallel workers."""
    # 1. Filter data for this specific user
    user_df = full_dataset[full_dataset['user_id'] == user_id].copy()

    if user_df.empty:
        return user_id, None

    # 2. Predict
    dmatrix = xgb.DMatrix(user_df[features])
    user_df['score'] = bst.predict(dmatrix)

    # 3. Format result: list of tuples (product_id, score)
    top_items = (
        user_df.sort_values(by='score', ascending=False)
        .head(10)[['product_id', 'score']]
        .to_records(index=False)
        .tolist()
    )
    return user_id, top_items


In [21]:
import logging
import pandas as pd
from typing import List, Dict
from joblib import Parallel, delayed
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def recommend_for_users_parallel(
    user_ids: List[int],
    full_dataset: pd.DataFrame,
    model_path: str,
    features: List[str],
    n_jobs: int = -1
) -> Dict[int, list]:
    """
    Parallel recommendation generation with clear and specific error handling.
    """
    if not user_ids:
        raise ValueError("user_ids list cannot be empty")

    if full_dataset is None or full_dataset.empty:
        raise ValueError("full_dataset is empty or None")

    if not features:
        raise ValueError("features list cannot be empty")

    try:
        # 1. Load model ONCE in main process
        logger.info(f"Loading XGBoost model from: {model_path}")
        bst = load_xgb(model_path)
        bst.set_param('nthread', 1)

        logger.info(f"Starting parallel processing for {len(user_ids)} users with {n_jobs} jobs...")

        # 2. Parallel execution with tqdm
        results = Parallel(
            n_jobs=n_jobs,
            backend='loky',
            verbose=0
        )(
            delayed(process_single_user_ltr)(uid, full_dataset, bst, features)
            for uid in tqdm(user_ids, desc="Generating Recommendations", unit="user")
        )

        # 3. Process results
        recommendation_dict = dict()
        success_count = int()
        error_count = int()

        for uid, recs in results:
            if recs is not None and len(recs) > 0:
                recommendation_dict[uid] = recs
                success_count += 1
            else:
                logger.warning(f"No recommendations generated for user {uid}")
                error_count += 1

        logger.info(f"Completed -> {success_count}/{len(user_ids)} users successful | "
                   f"{error_count} failed")

        if success_count == 0:
            raise RuntimeError("All users failed to generate recommendations. Check process_single_user_ltr function.")

        return recommendation_dict

    except FileNotFoundError:
        raise FileNotFoundError(f"Model file not found at path: {model_path}") from None
    except ValueError as ve:
        raise ValueError(f"Invalid input data: {str(ve)}") from ve
    except ImportError:
        raise ImportError("Required libraries (xgboost/joblib) are not installed properly") from None
    except Exception as e:
        logger.error(f"Unexpected error in parallel processing: {str(e)}", exc_info=True)
        raise RuntimeError(f"Failed to generate recommendations: {str(e)}") from e


In [22]:
# Optional: Sequential version for debugging
def recommend_for_users_sequential(
    user_ids: List[int],
    full_dataset: pd.DataFrame,
    model_path: str,
    features: List[str]
) -> Dict[int, list]:
    bst = load_xgb(model_path)
    bst.set_param('nthread', 1)

    recommendation_dict = {}

    for uid in tqdm(user_ids, desc="Sequential Processing", unit="user"):
        try:
            recs = process_single_user_ltr(uid, full_dataset, bst, features)
            if recs is not None and len(recs) > 0:
                recommendation_dict[uid] = recs
        except Exception as e:
            logger.error(f"Failed for user {uid}: {e}")

    return recommendation_dict

In [23]:
import pandas as pd
import numpy as np

def get_readable_recommendations(rec_dict, data_product_df):
    """
    Converts the recommendation dictionary into a formatted DataFrame
    with Product Names, Magnitudes, and Remark column.
    """
    # 1. Flatten the dictionary into a list of rows
    # rec_dict format: {user_id: [(prod_id, score), ...]}
    rows = list()
    for user_id, recs in rec_dict.items():
        for prod_id, score in recs:
            rows.append({
                "userID": user_id,
                "productID": prod_id,
                "Magnitude": score
            })

    # 2. Create a temporary DataFrame from the results
    df_results = pd.DataFrame(rows)

    # 3. Merge with DataProduct to get names
    final_df = df_results.merge(
        data_product_df[['product_id', 'product_name']],
        left_on='productID',
        right_on='product_id',
        how='left'
    )

    # 4. Cleanup and Rename columns
    final_df = final_df.rename(columns={'product_name': 'productName'})

    # 5. Add Remark Column
    conditions = [
        final_df["Magnitude"] >= 0.2,
        (final_df["Magnitude"] > 0.0) & (final_df["Magnitude"] < 0.2),
        final_df["Magnitude"] <= 0.0
    ]

    choices = [
        "intention to buy again",
        "not really necessary",
        "I hate it!"
    ]

    final_df["Remark"] = np.select(conditions, choices, default="not really necessary")

    # 6. Reorder columns
    final_df = final_df[
        ["userID", "productID", "productName", "Magnitude", "Remark"]
    ]

    # 7. Sort
    final_df = final_df.sort_values(
        by=['userID', 'Magnitude'],
        ascending=[True, False]
    )
    return final_df

In [24]:
from pathlib import Path

dir_path = Path("/content/drive/MyDrive/Colab Notebooks")
model_filename = 'xgboost_ltr_model.json'
MFP = dir_path / model_filename

In [25]:
# the usage of Features
TheFeature = ['order_number', 'order_dow', 'days_since_prior_order',
       'add_to_cart_order', 'user_total_orders', 'user_avg_days_between',
       'user_avg_cart_pos', 'user_total_reorders', 'prod_total_reorders',
       'prod_reorder_rate', 'prod_order_count', 'prod_avg_cart_pos']

In [26]:
user_list = [110001, 42214]
final_recs = recommend_for_users_parallel(user_list, test_df, MFP, TheFeature)

Generating Recommendations: 100%|██████████| 2/2 [00:00<00:00, 534.54user/s]


In [27]:
print(final_recs)

{110001: [(21137, 2.193127393722534), (26209, 1.5212301015853882), (2781, 1.0138267278671265), (21616, 0.7028533816337585), (33731, 0.43611109256744385), (36070, 0.001868915162049234), (27156, -0.04143115133047104), (41588, -0.3019513189792633), (13517, -0.47096434235572815), (49628, -0.8822891116142273)], 42214: [(24838, 1.2657248973846436), (46667, -0.07607019692659378), (23178, -0.3764210045337677)]}


In [28]:
final_report = get_readable_recommendations(final_recs, DataProduct)
display(final_report)

,userID,productID,productName,Magnitude,Remark
10,42214,24838,Unsweetened Almondmilk,1.27,intention to buy again
11,42214,46667,Organic Ginger Root,-0.08,I hate it!
12,42214,23178,Pure Lemon Juice,-0.38,I hate it!
0,110001,21137,Organic Strawberries,2.19,intention to buy again
1,110001,26209,Limes,1.52,intention to buy again
2,110001,2781,Chipotle Lime Meat-Free Crispy Fingers,1.01,intention to buy again
3,110001,21616,Organic Baby Arugula,0.70,intention to buy again
4,110001,33731,Grated Parmesan,0.44,intention to buy again
5,110001,36070,"Super Spinach! Baby Spinach, Baby Bok Choy, Sw...",0.00,not really necessary
6,110001,27156,Organic Black Beans,-0.04,I hate it!


## Fill gap 10 item and removed *"Hated items"* by *"might be interested"* category

In [29]:
import logging
import pandas as pd
import duckdb
from typing import List, Optional

# ====================== LOGGING SETUP ======================
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


In [30]:
def setup_duckdb(ltr_predictions: pd.DataFrame,
                 full_features_df: pd.DataFrame,
                 products_df: pd.DataFrame
                ) -> duckdb.DuckDBPyConnection:
    con = None
    try:
        logger.debug("Setting up DuckDB...")
        con = duckdb.connect(database=':memory:')
        con.execute("CREATE TABLE ltr AS SELECT * FROM ltr_predictions")
        con.execute("CREATE TABLE features AS SELECT * FROM full_features_df")
        con.execute("CREATE TABLE products AS SELECT * FROM products_df")
        logger.info(f"DuckDB ready → ltr:{len(ltr_predictions)} | features:{len(full_features_df)} | products:{len(products_df)}")

    except duckdb.Error as e:
        logger.error(f"Database engine error: {e}", exc_info=True)
        # Re-raising the specific DB error is often better than a generic ValueError
        raise

    except Exception as e:
        logger.critical(f"Unexpected error during DuckDB setup: {e}", exc_info=True)
        raise RuntimeError(f"System failure during database setup: {e}") from e

    finally:
        return con

In [31]:
def get_current_recommendations(
    con: duckdb.DuckDBPyConnection,
    user_id: int
) -> pd.DataFrame:
    try:
        logger.debug(f"Fetching current LTR recs for user {user_id}")
        return con.execute("""
            SELECT * FROM ltr
            WHERE userID = ?
            ORDER BY Magnitude DESC
        """, [user_id]).df()
    except Exception as e:
        logger.error(f"Error fetching current recs for user {user_id}: {str(e)}", exc_info=True)
        return pd.DataFrame()

In [32]:
def get_current_recommendations(con: duckdb.DuckDBPyConnection,
                                user_id: int,
                               ) -> pd.DataFrame:
    try:
        return con.execute("SELECT * FROM ltr WHERE userID = ? ORDER BY Magnitude DESC", [user_id]).df()
    except Exception as e:
        logger.error(f"Error fetching recs for user {user_id}: {str(e)}", exc_info=True)
        return pd.DataFrame()

In [33]:
def filter_high_quality(current_recs: pd.DataFrame,
                        threshold: float = 0.1,
                       ) -> pd.DataFrame:
    try:
        if current_recs.empty:
            return pd.DataFrame()
        return current_recs[current_recs['Magnitude'] > threshold].copy()
    except Exception as e:
        logger.error(f"Error filtering high-quality: {str(e)}", exc_info=True)
        return pd.DataFrame()


In [34]:
def get_user_profile(con: duckdb.DuckDBPyConnection, user_id: int) -> Optional[pd.Series]:
    try:
        df = con.execute("""
            SELECT
                aisle_id,
                department_id,
                days_since_prior_order,
                add_to_cart_order,
                reordered,
                user_total_orders,
                user_avg_days_between,
                user_avg_cart_pos,
                user_total_reorders
            FROM
                features
            WHERE
                user_id = ?
            ORDER BY
                order_number DESC LIMIT 1
        """, [user_id]).df()
        return df.iloc[0] if not df.empty else None
    except Exception as e:
        logger.error(f"Error getting profile for user {user_id}: {str(e)}", exc_info=True)
        return None

In [35]:
def find_primary_fillers(
    con: duckdb.DuckDBPyConnection,
    user_id: int,
    user_profile: pd.Series,
    slots_needed: int
) -> pd.DataFrame:
    """First tier: strict similar products"""
    try:
        if user_profile is None:
            return pd.DataFrame()

        df = con.execute("""
        WITH
            max_reorders AS (
            SELECT
                MAX(prod_total_reorders) AS max_val
            FROM
                features
            ),

            user_profile AS (
            SELECT
                ? AS aisle_id,
                ? AS department_id,
                ? AS days_since,
                ? AS cart_pos,
                ? AS reordered,
                ? AS total_orders,
                ? AS avg_days,
                ? AS avg_cart,
                ? AS total_reorders
                )
            SELECT
                f.product_id AS productID,
                p.product_name AS productName,
                (MAX(f.prod_total_reorders) * 1.0 / NULLIF((SELECT max_val FROM max_reorders), 0) * 100) AS Magnitude
            FROM
                features AS f
            INNER JOIN
                products AS p
            ON
                f.product_id = p.product_id
            WHERE
                f.aisle_id = (SELECT aisle_id FROM user_profile)
                AND f.department_id = (SELECT department_id FROM user_profile)
                AND ABS(f.days_since_prior_order - (SELECT days_since FROM user_profile)) <= 2
                AND ABS(f.add_to_cart_order - (SELECT cart_pos FROM user_profile)) <= 3
                AND ABS(f.user_total_orders - (SELECT total_orders FROM user_profile)) <= 2
                AND ABS(f.user_avg_days_between - (SELECT avg_days FROM user_profile)) <= 3
                AND f.product_id NOT IN (SELECT productID FROM ltr WHERE userID = ?)
            GROUP BY
                f.product_id, p.product_name
            ORDER BY
                MAX(f.prod_total_reorders) DESC
            LIMIT ?
        """, [
            int(user_profile['aisle_id']),
            int(user_profile['department_id']),
            int(user_profile['days_since_prior_order']),
            int(user_profile['add_to_cart_order']),
            int(user_profile['reordered']),
            int(user_profile['user_total_orders']),
            float(user_profile['user_avg_days_between']),
            float(user_profile['user_avg_cart_pos']),
            int(user_profile['user_total_reorders']),
            user_id,
            slots_needed + 3
        ]).df()

        return df.head(slots_needed)
    except Exception as e:
        logger.error(f"Error in primary fillers for user {user_id}: {str(e)}", exc_info=True)
        return pd.DataFrame()

In [36]:
def find_fallback_fillers(
    con: duckdb.DuckDBPyConnection,
    user_id: int,
    user_profile: pd.Series,
    slots_needed: int
) -> pd.DataFrame:
    """Second tier: same aisle + department only (looser constraints)"""
    try:
        if user_profile is None:
            return pd.DataFrame()

        df = con.execute("""
            WITH max_reorders AS (SELECT MAX(prod_total_reorders) AS max_val FROM features),
                 user_profile AS (
                    SELECT ? AS aisle_id, ? AS department_id
                 )
            SELECT
                f.product_id AS productID,
                p.product_name AS productName,
                (MAX(f.prod_total_reorders) * 1.0 / NULLIF((SELECT max_val FROM max_reorders), 0) * 100) AS Magnitude
            FROM features f
            INNER JOIN products p ON f.product_id = p.product_id
            WHERE f.aisle_id = (SELECT aisle_id FROM user_profile)
              AND f.department_id = (SELECT department_id FROM user_profile)
              AND f.product_id NOT IN (SELECT productID FROM ltr WHERE userID = ?)
            GROUP BY f.product_id, p.product_name
            ORDER BY MAX(f.prod_total_reorders) DESC
            LIMIT ?
        """, [
            int(user_profile['aisle_id']), int(user_profile['department_id']),
            user_id, slots_needed + 5
        ]).df()

        return df.head(slots_needed)
    except Exception as e:
        logger.error(f"Error in fallback fillers for user {user_id}: {str(e)}", exc_info=True)
        return pd.DataFrame()

In [37]:
def find_similar_popular_fillers(
    con: duckdb.DuckDBPyConnection,
    user_id: int,
    user_profile: pd.Series,
    slots_needed: int,
    buffer: int = 5
) -> pd.DataFrame:
    """
    FIXED: Proper normalized Magnitude (0.01 - 0.19 range)
    Uses real product_name and avoids obsolete products.
    """
    try:
        if user_profile is None:
            logger.warning(f"No profile for user {user_id}")
            return pd.DataFrame()

        logger.debug(f"Finding {slots_needed} normalized fillers for user {user_id}")

        df = con.execute("""
            WITH max_reorders AS (
                SELECT MAX(prod_total_reorders) AS max_val
                FROM features
            ),
            user_profile AS (
                SELECT
                    ? AS aisle_id,
                    ? AS department_id,
                    ? AS days_since,
                    ? AS cart_pos,
                    ? AS reordered,
                    ? AS total_orders,
                    ? AS avg_days,
                    ? AS avg_cart,
                    ? AS total_reorders
            )
            SELECT
                f.product_id AS productID,
                p.product_name AS productName,
                (MAX(f.prod_total_reorders) * 1.0 / NULLIF((SELECT max_val FROM max_reorders), 0) * 100) AS Magnitude
            FROM features f
            INNER JOIN products p
                ON f.product_id = p.product_id
            WHERE f.aisle_id = (SELECT aisle_id FROM user_profile)
              AND f.department_id = (SELECT department_id FROM user_profile)
              AND ABS(f.days_since_prior_order - (SELECT days_since FROM user_profile)) <= 2
              AND ABS(f.add_to_cart_order - (SELECT cart_pos FROM user_profile)) <= 3
              AND ABS(f.user_total_orders - (SELECT total_orders FROM user_profile)) <= 2
              AND ABS(f.user_avg_days_between - (SELECT avg_days FROM user_profile)) <= 3
              AND f.product_id NOT IN (
                  SELECT productID FROM ltr WHERE userID = ?
              )
            GROUP BY f.product_id, p.product_name
            ORDER BY MAX(f.prod_total_reorders) DESC
            LIMIT ?
        """, [
            int(user_profile['aisle_id']),
            int(user_profile['department_id']),
            int(user_profile['days_since_prior_order']),
            int(user_profile['add_to_cart_order']),
            int(user_profile['reordered']),
            int(user_profile['user_total_orders']),
            float(user_profile['user_avg_days_between']),
            float(user_profile['user_avg_cart_pos']),
            int(user_profile['user_total_reorders']),
            user_id,
            slots_needed + buffer
        ]).df()

        logger.debug(f"Found {len(df)} fillers for user {user_id}")
        return df.head(slots_needed)

    except Exception as e:
        logger.error(f"Error finding fillers for user {user_id}: {str(e)}", exc_info=True)
        return pd.DataFrame()

In [38]:
def build_final_recs_for_user(
    high_quality: pd.DataFrame,
    fillers: pd.DataFrame,
    target_n: int,
    user_id: int
) -> pd.DataFrame:
    try:
        if high_quality.empty and fillers.empty:
            logger.warning(f"No recs possible for user {user_id}")
            return pd.DataFrame()

        if not fillers.empty:
            fillers = fillers.copy()
            fillers['Remark'] = 'similar popular candidate (filler)'

        combined = pd.concat([high_quality, fillers], ignore_index=True)
        final = combined.head(target_n).drop_duplicates(subset=['productID'])

        # Final safety padding with best available if still short
        if len(final) < target_n:
            logger.warning(f"Only {len(final)} recs for user {user_id} → using extra fallback")
            # You can extend further if needed

        final['userID'] = user_id
        return final
    except Exception as e:
        logger.error(f"Error building final recs for user {user_id}: {str(e)}", exc_info=True)
        return pd.DataFrame()

In [39]:
# ====================== MAIN ORCHESTRATOR ======================

def generate_final_recommendations(
    ltr_predictions: pd.DataFrame,
    full_features_df: pd.DataFrame,
    products_df: pd.DataFrame,
    target_n: int = 10
) -> pd.DataFrame:
    con = None
    try:
        logger.info(f"Starting final recommendation generation for {ltr_predictions['userID'].nunique()} users...")

        con = setup_duckdb(ltr_predictions, full_features_df, products_df)

        final_results: List[pd.DataFrame] = list()
        user_ids = sorted(ltr_predictions['userID'].unique())

        for user_id in user_ids:
            try:
                current = get_current_recommendations(con, int(user_id))
                high_quality = filter_high_quality(current)
                slots_needed = target_n - len(high_quality)

                if slots_needed <= 0:
                    final_for_user = high_quality.head(target_n).copy()
                    final_for_user['Remark'] = final_for_user.get('Remark', 'intention to buy again')
                else:
                    profile = get_user_profile(con, int(user_id))

                    # Try primary (strict similarity)
                    fillers = find_primary_fillers(con, int(user_id), profile, slots_needed)

                    # If still not enough -> use fallback (2nd largest magnitude logic)
                    if len(fillers) < slots_needed:
                        remaining = slots_needed - len(fillers)
                        fallback = find_fallback_fillers(con, int(user_id), profile, remaining)
                        fillers = pd.concat([fillers, fallback], ignore_index=True).head(slots_needed)

                    final_for_user = build_final_recs_for_user(
                        high_quality=high_quality,
                        fillers=fillers,
                        target_n=target_n,
                        user_id=int(user_id)
                    )

                final_results.append(final_for_user)

            except Exception as e_user:
                logger.error(f"Failed user {user_id}: {str(e_user)}")
                final_results.append(pd.DataFrame([{
                    'userID': user_id, 'productID': -1, 'productName': 'ERROR',
                    'Magnitude': 0.0, 'Remark': 'error'
                }]))

        result_df = pd.concat(final_results, ignore_index=True)
        result_df = result_df.groupby('userID').head(target_n)

        logger.info(f"Completed successfully! {result_df['userID'].nunique()} users × {target_n} recs")
        return result_df[['userID', 'productID', 'productName', 'Magnitude', 'Remark']]

    except Exception as e:
        logger.error(f"Critical error: {str(e)}", exc_info=True)
        raise
    finally:
        if con is not None:
            con.close()

In [40]:
double_final_df      = generate_final_recommendations(
    ltr_predictions  = final_report,
    full_features_df = test_df,
    products_df      = DataProduct,
    target_n         = 10,
)

In [41]:
display(double_final_df)

,userID,productID,productName,Magnitude,Remark
0,42214,24838,Unsweetened Almondmilk,1.27,intention to buy again
1,42214,2966,Pure Coconut Water,2.83,similar popular candidate (filler)
2,42214,18811,Organic Apple Juice,1.99,similar popular candidate (filler)
3,42214,19173,Orange Calcium & Vitamin D Pulp Free,1.75,similar popular candidate (filler)
4,42214,14715,Coconut Water,1.58,similar popular candidate (filler)
5,42214,30639,Organic Orange Juice,1.58,similar popular candidate (filler)
6,42214,32156,Cranberry Juice Cocktail,0.92,similar popular candidate (filler)
7,42214,13259,Organic Variety Pack,0.83,similar popular candidate (filler)
8,42214,24850,Organic Super Fruit Punch Juice Drink,0.78,similar popular candidate (filler)
9,42214,34335,Ruby Red Grapefruit Juice,0.71,similar popular candidate (filler)


## Parallel Processing version

In [42]:
from joblib import Parallel, delayed
from typing import Tuple
from tqdm import tqdm

# ====================== WORKER ======================
def process_single_user_filler(task: Tuple) -> pd.DataFrame: # Renamed
    user_id, ltr_df, features_df, products_df, target_n = task
    con = None
    try:
        con = setup_duckdb(ltr_df, features_df, products_df)

        current = get_current_recommendations(con, int(user_id))
        high_quality = filter_high_quality(current)
        slots_needed = target_n - len(high_quality)

        if slots_needed <= 0:
            final_for_user = high_quality.head(target_n).copy()
            final_for_user['Remark'] = final_for_user.get('Remark', 'intention to buy again')
        else:
            profile = get_user_profile(con, int(user_id))
            fillers = find_primary_fillers(con, int(user_id), profile, slots_needed)

            if len(fillers) < slots_needed:
                remaining = slots_needed - len(fillers)
                fallback = find_fallback_fillers(con, int(user_id), profile, remaining)
                fillers = pd.concat([fillers, fallback], ignore_index=True).head(slots_needed)

            final_for_user = build_final_recs_for_user(high_quality, fillers, target_n, int(user_id))

        return final_for_user

    except Exception as e:
        logger.error(f"User {user_id} crashed: {str(e)}", exc_info=True)
        return pd.DataFrame([{'userID': user_id, 'productID': -1, 'productName': 'ERROR',
                              'Magnitude': 0.0, 'Remark': 'error'}])
    finally:
        if con is not None:
            con.close()

In [49]:
# ====================== MAIN FUNCTION ======================
def generate_final_recommendations_multiprocess(
    ltr_predictions: pd.DataFrame,
    full_features_df: pd.DataFrame,
    products_df: pd.DataFrame,
    target_n: int = 10,
    n_jobs: int = -1
) -> pd.DataFrame:

    logger.info(f"Starting joblib parallel processing for {ltr_predictions['userID'].nunique()} users...")

    if n_jobs == -1:
        n_jobs = max(1, os.cpu_count() or 2)

    user_ids = sorted(ltr_predictions['userID'].unique())
    tasks = [(uid, ltr_predictions, full_features_df, products_df, target_n) for uid in user_ids]

    results = Parallel(n_jobs=n_jobs, backend='loky')(
        delayed(process_single_user_filler)(task) # Updated to new name
        for task in tqdm(tasks, desc="Filling the Gaps", unit="user")
    )

    result_df = pd.concat(results, ignore_index=True)
    result_df = result_df.groupby('userID').head(target_n)

    logger.info(f"DONE! {result_df['userID'].nunique()} users with {target_n} recs each")
    return result_df[['userID', 'productID', 'productName', 'Magnitude', 'Remark']]

In [44]:
third_final_df = generate_final_recommendations_multiprocess(
    ltr_predictions=final_report,
    full_features_df=test_df,
    products_df=DataProduct,
    target_n=10,
    n_jobs=-1,
)

Generating Recommendations: 100%|██████████| 2/2 [00:00<00:00, 144.45user/s]


In [45]:
third_final_df

,userID,productID,productName,Magnitude,Remark
0,42214,24838,Unsweetened Almondmilk,1.27,intention to buy again
1,42214,2966,Pure Coconut Water,2.83,similar popular candidate (filler)
2,42214,18811,Organic Apple Juice,1.99,similar popular candidate (filler)
3,42214,19173,Orange Calcium & Vitamin D Pulp Free,1.75,similar popular candidate (filler)
4,42214,30639,Organic Orange Juice,1.58,similar popular candidate (filler)
5,42214,14715,Coconut Water,1.58,similar popular candidate (filler)
6,42214,32156,Cranberry Juice Cocktail,0.92,similar popular candidate (filler)
7,42214,13259,Organic Variety Pack,0.83,similar popular candidate (filler)
8,42214,24850,Organic Super Fruit Punch Juice Drink,0.78,similar popular candidate (filler)
9,42214,34335,Ruby Red Grapefruit Juice,0.71,similar popular candidate (filler)


In [50]:
# Stress Test

from time import time

start = time()

all_userID = test_df['user_id'].unique().tolist()
testID = all_userID[:500]
stress_final_recs_raw = recommend_for_users_parallel(testID, test_df, MFP, TheFeature)
stress_final_recs = get_readable_recommendations(stress_final_recs_raw, DataProduct)
stress_final_df = generate_final_recommendations_multiprocess(
    ltr_predictions=stress_final_recs,
    full_features_df=test_df,
    products_df=DataProduct,
    target_n=10,
    n_jobs=-1,
)

finaltime = time()
print(f'The time consume as {round((finaltime - start)/60, 2):,} minutes.')

Filling the Gaps: 100%|██████████| 500/500 [02:58<00:00,  2.80user/s]


The time consume as 3.07 minutes.


# Conclusion

The implementation of the **Learning-to-Rank (LTR) model using XGBoost** successfully ranks products based on predicted relevance for each user. However, the experiment shows that the model only produced **three recommendations** for user **42214**, even though the system requested **Top-10 recommendations**.

This occurred because the **candidate items provided to the LTR model only included products previously purchased by the user**. Since Learning-to-Rank models such as XGBoost **do not generate new items**, but instead **rank only the items provided as input**, the model could only rank the three available products.

Therefore, the limitation observed is **not caused by the XGBoost LTR model itself**, but rather by the **candidate generation stage**, which restricts the available items to rank. This highlights an important architectural principle in recommender systems:

**A production-level recommender system must consist of two main components:**

1. Candidate Generation (Generate possible items)
2. Ranking Model (LTR using XGBoost)

Without a proper candidate generation stage, the Learning-to-Rank model cannot produce sufficient recommendations, even when the ranking model performs well.

Thus, the experiment demonstrates that **Learning-to-Rank models are highly effective for ranking tasks**, but **must be combined with candidate generation strategies** to produce meaningful Top-N recommendations.

---

In [57]:
import pandas as pd
from typing import Optional

def get_remark_distribution(df: pd.DataFrame,
                           column: str = 'Remark',
                           sort_by: str = 'Count',
                           ascending: bool = False,
                           ) -> pd.DataFrame:
    if df is None or df.empty:
        raise ValueError("Input DataFrame is empty or None")

    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in DataFrame")

    # Calculate value counts
    stats = df[column].value_counts().reset_index()
    stats.columns = ['Remark', 'Count']

    # Calculate percentage
    total = stats['Count'].sum()
    stats['Percentage'] = (stats['Count'] / total * 100).round(2)
    stats['Percentage_str'] = stats['Percentage'].astype(str) + '%'

    # Calculate cumulative percentage
    stats['Cumulative_%'] = stats['Percentage'].cumsum().round(2)
    stats['Cumulative_%_str'] = stats['Cumulative_%'].astype(str) + '%'

    # Sort
    stats = stats.sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)

    # Reorder columns nicely
    final_stats = stats[['Remark', 'Count', 'Percentage', 'Percentage_str',
                        'Cumulative_%', 'Cumulative_%_str']]
    print(f"Remark Distribution Analysis - Total Records: {total:,}")
    return final_stats


# ========================== USAGE EXAMPLE ==========================

def show_remark_stats(stress_final_df: pd.DataFrame):
    """Convenient wrapper to display nicely"""
    stats_df = get_remark_distribution(stress_final_df)

    # Pretty display in Jupyter
    display(stats_df.style.set_caption("Remark Distribution with Percentage"))
    return stats_df

In [56]:
remark_stats = show_remark_stats(stress_final_df)

Remark Distribution Analysis - Total Records: 4,973


,Remark,Count,Percentage,Percentage_str,Cumulative_%,Cumulative_%_str
0,similar popular candidate (filler),3049,61.310000,61.31%,61.310000,61.31%
1,intention to buy again,1751,35.210000,35.21%,96.520000,96.52%
2,not really necessary,173,3.480000,3.48%,100.000000,100.0%


# Suggestions

Based on the findings, the following improvements are recommended to enhance the recommendation system:

## 1. Implement Candidate Generation Before LTR

To ensure sufficient items for ranking, a candidate generation module should be added before the XGBoost LTR model. This module should generate **100–500 candidate items** per user before ranking.

This approach is feasible and commonly used in production recommender systems.

---

## 2. Use Popular Item Candidate Generation

Popular products can be recommended to users who have not recently purchased them. This method is simple, fast, and effective, especially for cold-start situations.

Advantages:

* Easy to implement
* Always produces enough candidates
* Works well for small datasets

This approach can be implemented using product purchase frequency.

---

## 3. Use Similar User (Collaborative Filtering) Candidate Generation

Another approach is recommending items purchased by users with similar purchasing behavior. This method improves personalization and helps discover new products.

Advantages:

* Personalized recommendation
* Introduces new products
* Suitable for e-commerce datasets

This method can be implemented using user-item interaction matrices.

---

## 4. Use Aisle or Department-Based Expansion

Since the dataset contains **aisle_id** and **department_id**, recommendations can be generated from categories previously purchased by the user.

Advantages:

* Strong performance in grocery datasets
* Simple feature expansion
* Low computational cost

This approach is highly feasible and suitable for Instacart-like datasets.

---

## 5. Combine Multiple Candidate Generation Methods (Hybrid Approach)

The best approach is to combine multiple strategies:

* Popular products
* Similar users
* Same aisle or department
* Previously purchased items

This hybrid method improves recommendation diversity and accuracy.

---

## 6. Maintain Two-Stage Recommendation Architecture

A production-ready system should follow:

Stage 1 — Candidate Generation
Stage 2 — XGBoost Learning-to-Rank Model

This architecture ensures:

* Enough candidate items
* Better ranking quality
* Scalable recommendation system

---

## 7. Evaluate Using Ranking Metrics

After implementing candidate generation, the model performance should be evaluated using ranking metrics such as:

* NDCG@K
* MAP@K
* MRR

These metrics measure ranking quality rather than classification accuracy and are appropriate for Learning-to-Rank systems.

---

# Final Recommendation

To improve the recommendation system, it is strongly suggested to implement a **candidate generation module** before the **XGBoost Learning-to-Rank model**. This enhancement will enable the system to produce sufficient Top-N recommendations, improve personalization, and align the architecture with production-level recommender systems used in industry.

With this improvement, the recommendation system will become:

* More scalable
* More accurate
* More production-ready
